# Lighthouse: Exploratory Data Analysis (EDA)
**Author:** Leonardo Farias dos Santos  
**Objective:** Evaluate the integrity, distribution, and reliability of the raw dataset (vendas_2023_2024.csv) to support executive decision-making and enable future AI modeling.

In [ ]:
### config sql extension and load the database

In [1]:
%load_ext sql

%sql duckdb:///:memory:

%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False

Tip: You may define configurations in /home/leofariasrj25/01 - Projects/01 - Coding Projects/desafio_indicium/pyproject.toml or /home/leofariasrj25/.jupysql/config.

Did not find user configurations in /home/leofariasrj25/01 - Projects/01 - Coding Projects/desafio_indicium/pyproject.toml.

Connecting to 'duckdb:///:memory:'

## Part 1 and 2 - Dataset overview

In [ ]:
%%sql

CREATE VIEW sales AS
SELECT * FROM read_csv_auto('../data/raw/vendas_2023_2024.csv');
    
WITH sales_metadata AS (
    SELECT COUNT(*) AS column_count 
    FROM (DESCRIBE SELECT * FROM sales LIMIT 1)
),

sales_data AS (
    SELECT 
        COUNT(*) AS line_count,
        MIN(sale_date) AS min_date,
        MAX(sale_date) AS max_date,
        MIN(total) AS min_total,
        MAX(total) as max_total,
        CAST(AVG(total) AS DECIMAL(15,2)) as avg_total
    FROM sales
)

SELECT
    d.line_count AS "Total de Registros",
    m.column_count AS "Total de Colunas",
    d.min_date AS "Data Inicial",
    d.max_date AS "Data Final",
    d.min_total AS "Menor Venda (R$)",
    d.max_total AS "Maior Venda (R$)",
    d.avg_total AS "Ticket Médio (R$)"
FROM sales_data d
CROSS JOIN sales_metadata m;



## Part 3 - Analyzing Data Quality

### 1.3.1 - Verify sale discrepancy from highest to lowest.

In [12]:
%%sql

    
SELECT total
FROM sales
ORDER BY total DESC
LIMIT 10;

,total
0,2222973.00
1,2222973.00
2,2222973.00
3,2222973.00
4,2147399.00
5,2111824.35
6,2111824.35
7,2074775.00
8,2074775.00
9,2030026.00


## 1.3.2 - Verify sale discrepancy from low to high.

In [13]:
%%sql
    
SELECT total
FROM sales
ORDER BY total ASC
LIMIT 10;

,total
0,294.50
1,294.50
2,294.50
3,439.85
4,463.00
5,463.00
6,471.20
7,471.20
8,588.05
9,588.05


### 1.3.3 - Check if there are records with empty fields/columns


In [11]:
%%sql

    
SELECT * FROM sales WHERE id is NULL OR client_id is NULL total is NULL OR id is NULL OR sale_date IS NULL;

,id,id_client,id_product,qtd,total,sale_date


### 1.3.4 - Check if there are duplicate sales

In [7]:
%%sql

SELECT 
    id, 
    COUNT(*) AS frequency_count
FROM sales
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY frequency_count DESC
LIMIT 5;

,id,frequency_count
